# **Analyzing and Making Predictions on Insurance Data**

In [ ]:
# Setup

import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder


from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib as jlb 

In [ ]:
data = pd.read_csv('../data/insurance.csv')

**Data Analysis**

In [ ]:
display(data.sample(5))

display(data.info())

display(data.describe())

display(data.isnull().sum())

,age,sex,bmi,children,smoker,region,charges
832,28,female,23.845,2,no,northwest,4719.73655
372,42,female,33.155,1,no,northeast,7639.41745
676,55,female,40.810,3,no,southeast,12485.80090
619,55,female,37.100,0,no,southwest,10713.64400
720,51,female,40.660,0,no,northeast,9875.68040


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB


None

,age,bmi,children,charges
count,1338.000000,1338.000000,1338.000000,1338.000000
mean,39.207025,30.663397,1.094918,13270.422265
std,14.049960,6.098187,1.205493,12110.011237
min,18.000000,15.960000,0.000000,1121.873900
25%,27.000000,26.296250,0.000000,4740.287150
50%,39.000000,30.400000,1.000000,9382.033000
75%,51.000000,34.693750,2.000000,16639.912515
max,64.000000,53.130000,5.000000,63770.428010


age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

**Visualizations**

In [ ]:
# Distribution of charges
fig = px.histogram(data, x='charges', nbins=30, title='Distribution of Insurance Charges')
fig.show()

# Feature vs Charges
fig = px.scatter(data, x='bmi', y='charges', color='smoker', title='BMI vs Charges by Smoker')
fig.show()

**Numeric conversion**

In [ ]:
# Firstly lets get the categorical data unique values
display(data.sex.unique())
display(data.smoker.unique())
display(data.region.unique())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB


None

array(['female', 'male'], dtype=object)

array(['yes', 'no'], dtype=object)

array(['southwest', 'southeast', 'northwest', 'northeast'], dtype=object)

In [ ]:
# Lets apply encoding to the values
encoded_data = data.copy()
encoded_data['sex'] = LabelEncoder().fit_transform(encoded_data['sex'])
encoded_data['smoker'] = LabelEncoder().fit_transform(encoded_data['smoker'])
encoded_data = pd.get_dummies(encoded_data, columns=['region'])

encoded_data.sample(5)

,age,sex,bmi,children,smoker,charges,region_northeast,region_northwest,region_southeast,region_southwest
1014,38,0,27.600,0,0,5383.53600,False,False,False,True
831,36,0,25.840,0,0,5266.36560,False,True,False,False
745,50,0,30.115,1,0,9910.35985,False,True,False,False
64,20,0,22.420,0,1,14711.74380,False,True,False,False
739,29,1,35.500,2,1,44585.45587,False,False,False,True


**Model Building**

In [ ]:
X = encoded_data.drop('charges', axis=1)
y = encoded_data['charges']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=69)
model = DecisionTreeRegressor(
    max_depth=4,
    min_samples_split=5,
    random_state=69
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Mean Absolute Error:", mean_absolute_error(y_test, y_pred))
print("Mean Squared Error:", mean_squared_error(y_test, y_pred))
print("R^2 Score:", r2_score(y_test, y_pred))

Mean Absolute Error: 2994.3830344541384
Mean Squared Error: 26394624.030756015
R^2 Score: 0.8474104337173005


In [ ]:
# Save the model
jlb.dump(model, '../models/insurance_charges_model.joblib')

['../models/insurance_charges_model.joblib']

In [ ]:
# import the model and make predictions
loaded_model = jlb.load('../models/insurance_charges_model.joblib')

In [ ]:
def predict_insurance_charges (age, sex, bmi, children, smoker, region):
    input_data = pd.DataFrame({
        'age': [age],
        'sex': [LabelEncoder().fit_transform(pd.Series([sex]))[0]],
        'bmi': [bmi],
        'children': [children],
        'smoker': [LabelEncoder().fit_transform(pd.Series([smoker]))[0]],
        'region_northeast': [1 if region == 'northeast' else 0],
        'region_northwest': [1 if region == 'northwest' else 0],
        'region_southeast': [1 if region == 'southeast' else 0],
        'region_southwest': [1 if region == 'southwest' else 0],
    })
    prediction = loaded_model.predict(input_data)
    return prediction[0]


# Example prediction
predicted_charge = predict_insurance_charges(29, 'female', 26.2, 0, 'yes', 'southeast')
print(f"Predicted Insurance Charge: ${predicted_charge:.2f}")

Predicted Insurance Charge: $3262.29
